##### Complaint Generator

This notebook generates synthetic customer complaints from delivered orders using AI SQL functions

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
COMPLAINT_RATE = float(dbutils.widgets.get("COMPLAINT_RATE"))
LLM_MODEL = dbutils.widgets.get("LLM_MODEL")

In [ ]:
import os
import random

from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

# Complaint category weights
CATEGORIES = ["delivery_delay", "missing_items", "food_quality", "service_issue", "other"]
WEIGHTS = [0.40, 0.25, 0.20, 0.10, 0.05]


@F.udf(returnType=StringType())
def assign_category(order_id: str) -> str:
    """Assign category based on hash of order_id for deterministic but varied distribution."""
    seed = hash(order_id) % (2**31)
    rng = random.Random(seed)
    return rng.choices(CATEGORIES, weights=WEIGHTS, k=1)[0]


# ── Per-batch inference cap ───────────────────────────────────────────────────
# Why this exists: previously this stream piped every sampled delivered event
# straight through `ai_gen()` in a `selectExpr`, then `.toTable(...)`. With
# `availableNow=True` and a fresh deploy, the canonical replay (60× speed)
# leaves a multi-thousand-row backlog of `delivered` events on day one. At
# COMPLAINT_RATE=0.15 that's still hundreds of `ai_gen()` calls per micro-batch,
# which routinely blew the 10-min `timeout_seconds` budget.  The cron would
# then queue another run (now disabled), and the backlog kept growing forever.
#
# Same pattern used in jobs/complaint_agent_stream.ipynb: split each batch into
# a small "real" slice (calls ai_gen) and an "overflow" slice (deterministic
# templated text).  Caps wall time at roughly MAX_INFERENCES_PER_BATCH *
# worst_case_latency.
CHECKPOINT_PATH = f"/Volumes/{CATALOG}/complaints/checkpoints/complaint_generator"
MAX_INFERENCES_PER_BATCH = 50


def is_first_run():
    """True when no checkpoint exists yet — drives the fake-it-till-up backfill
    path.  Identical heuristic to the agent streams."""
    return not os.path.exists(CHECKPOINT_PATH) or len(os.listdir(CHECKPOINT_PATH)) == 0


# ── Deterministic fallback complaint templates ────────────────────────────────
# Used both during initial backfill (drain the historical backlog instantly)
# and as the per-batch overflow path beyond MAX_INFERENCES_PER_BATCH.  Texts
# are realistic but obviously not LLM-generated — that's fine: at demo time
# the LLM-generated rows ride on top of a base of templated rows.
FAKE_COMPLAINTS = {
    "delivery_delay": [
        "My order took way too long to arrive, the food was cold by the time it got here.",
        "I waited over an hour for my food, this is completely unacceptable.",
        "The delivery was extremely late and no one bothered to let me know.",
    ],
    "missing_items": [
        "Half my order is missing and I was still charged the full amount.",
        "The drinks and sides I paid for never arrived with the order.",
        "Several items were missing from my bag, this is the third time this month.",
    ],
    "food_quality": [
        "The food was cold and tasted stale, completely inedible.",
        "My meal was overcooked and dry, totally unenjoyable.",
        "The food quality was awful, nothing like what I expected.",
    ],
    "service_issue": [
        "The driver was rude and threw my bag on the doorstep.",
        "I tried calling customer service three times and got nowhere.",
        "Worst service I've ever had, no one will take responsibility.",
    ],
    "other": [
        "I had a bad experience with my order and want this resolved quickly.",
        "Something went wrong with my order and I'd like a refund.",
        "Disappointed with this whole experience, please get back to me.",
    ],
}


@F.udf(returnType=StringType())
def get_fake_complaint(order_id: str, category: str) -> str:
    """Deterministic fake complaint text keyed by (order_id, category).
    Cheap fast path for backfill + overflow rows."""
    pool = FAKE_COMPLAINTS.get(category, FAKE_COMPLAINTS["other"])
    seed = hash(f"{order_id}|{category}") % (2**31)
    return random.Random(seed).choice(pool)

In [ ]:
# Create schema and checkpoint volume
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.complaints")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.complaints.checkpoints")

In [ ]:
# Stream processing pipeline.
# Samples COMPLAINT_RATE of both historical and new delivered orders, but no
# longer calls `ai_gen()` inline — that happens (capped) inside process_batch.
# Wrapped in a builder so the retry path can pass startingVersion="latest"
# and skip a multi-million-row historical backfill after a table recreate.
def _build_complaints_source(starting_version=None):
    reader = spark.readStream
    if starting_version is not None:
        reader = reader.option("startingVersion", str(starting_version))
    return (
        reader
        .table(f"{CATALOG}.lakeflow.all_events")
        .filter("event_type = 'delivered'")
        .filter(F.rand() < COMPLAINT_RATE)
        .withColumn("complaint_id", F.expr("uuid()"))
        .withColumn("complaint_category", assign_category(F.col("order_id")))
        .withColumn("source_ts", F.current_timestamp())
        .select("complaint_id", "order_id", "complaint_category", "source_ts")
    )


complaints_source = _build_complaints_source()


def process_batch(batch_df, batch_id):
    """Process each micro-batch with inference capping.

    - First run (no checkpoint): ALL rows get templated complaints (fast
      backfill — drains the entire historical backlog without an LLM call).
    - Subsequent runs: first MAX_INFERENCES_PER_BATCH rows get real `ai_gen()`
      output, rest fall back to templated text.  Bounds per-run wall time so
      a backlog cannot blow the 10-min task timeout.
    """
    if batch_df.isEmpty():
        return

    first_run = is_first_run()
    row_count = batch_df.count()

    print(f"Processing batch {batch_id}: {row_count} rows, first_run={first_run}")

    target = f"{CATALOG}.complaints.raw_complaints"

    if first_run:
        print(f"  -> First run detected, using templated complaints for all {row_count} rows")
        result_df = batch_df.select(
            F.col("complaint_id"),
            F.col("order_id"),
            F.current_timestamp().alias("ts"),
            F.col("complaint_category"),
            get_fake_complaint(F.col("order_id"), F.col("complaint_category")).alias("complaint_text"),
            F.lit("template_backfill").alias("generated_by"),
        )
        result_df.write.mode("append").saveAsTable(target)
        return

    # Stable order so the cap is deterministic and resumable.
    windowed = batch_df.withColumn(
        "row_num",
        F.row_number().over(Window.orderBy(F.col("source_ts"), F.col("complaint_id"))),
    )

    real_count = min(row_count, MAX_INFERENCES_PER_BATCH)
    fake_count = max(0, row_count - MAX_INFERENCES_PER_BATCH)
    print(f"  -> Real ai_gen(): {real_count} rows, templated: {fake_count} rows")

    real_df = (
        windowed
        .filter(f"row_num <= {MAX_INFERENCES_PER_BATCH}")
        .selectExpr(
            "complaint_id",
            "order_id",
            "current_timestamp() as ts",
            "complaint_category",
            """ai_gen(
                concat(
                    'You are an upset customer writing a complaint about your food delivery. ',
                    'Write a realistic 1-2 sentence complaint about: ',
                    complaint_category,
                    '. Order ID: ', order_id,
                    '. Be specific and sound frustrated but realistic. Do not include greeting or signature.'
                )
            ) as complaint_text""",
            "'llm_generator' as generated_by",
        )
    )

    fake_df = (
        windowed
        .filter(f"row_num > {MAX_INFERENCES_PER_BATCH}")
        .select(
            F.col("complaint_id"),
            F.col("order_id"),
            F.current_timestamp().alias("ts"),
            F.col("complaint_category"),
            get_fake_complaint(F.col("order_id"), F.col("complaint_category")).alias("complaint_text"),
            F.lit("template_overflow").alias("generated_by"),
        )
    )

    real_df.union(fake_df).write.mode("append").saveAsTable(target)

In [ ]:
# Create target table schema
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.complaints.raw_complaints (
  complaint_id STRING,
  order_id STRING,
  ts TIMESTAMP,
  complaint_category STRING,
  complaint_text STRING,
  generated_by STRING
)
""")

In [ ]:
# Write stream to raw_complaints table via foreachBatch so process_batch can
# apply the per-batch inference cap (see the cell that defines
# MAX_INFERENCES_PER_BATCH).
#
# The checkpoint lives in a UC volume that survives catalog/table rebuilds,
# so if the source Delta table (`all_events`) is dropped and recreated by
# the lakeflow pipeline, the checkpoint will still reference the old
# table id and Spark will refuse to read with:
#   DIFFERENT_DELTA_TABLE_READ_BY_STREAMING_SOURCE
# Recover by clearing the stale checkpoint and retrying once.
def _run_stream(stream_df):
    q = (
        stream_df.writeStream
        .foreachBatch(process_batch)
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .start()
    )
    q.awaitTermination()


try:
    _run_stream(complaints_source)
except Exception as e:
    if "DIFFERENT_DELTA_TABLE_READ_BY_STREAMING_SOURCE" in str(e):
        print(
            f"⚠️ Source Delta table id changed (table was recreated). "
            f"Clearing stale checkpoint at {CHECKPOINT_PATH} and restarting "
            f"from the LATEST version of the source (historical backfill skipped)."
        )
        dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)
        print(f"✅ Cleared {CHECKPOINT_PATH}")
        _run_stream(_build_complaints_source(starting_version="latest"))
    else:
        raise